# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector, built to be leakage-safe from the start, not cleaned up after the fact.**

Two groups of columns are deliberately left out here rather than filtered out later in Section 3: `trend_direction`/`trend_pct` (the label is computed FROM them, see ML-03), and the `_last_30d` traffic columns (that's the label's own comparison window, not a predictor). The 90-day aggregates (`impressions_90d`, `ctr`, `avg_position`, etc.) are also excluded for the same window-overlap reason, the skill's rule is "only the earlier sub-window is a legal feature," and `_prev_30d` is that earlier sub-window.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

static_numeric = ["word_count", "char_count", "search_volume", "competition", "cpc",
                   "content_age_days", "days_since_last_update"]
static_categorical = ["content_type", "main_intent", "competition_level",
                       "age_tier", "freshness_tier"]
safe_traffic = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

numeric_cols = static_numeric + safe_traffic
X_numeric = df[numeric_cols].fillna(-1)
X_categorical = pd.get_dummies(df[static_categorical].fillna("unknown"), prefix=static_categorical)

X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

print("feature vector shape:", X.shape)
print("columns:", X.columns.tolist())

feature vector shape: (30000, 30)
columns: ['word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'content_age_days', 'days_since_last_update', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown', 'competition_level_HIGH', 'competition_level_LOW', 'competition_level_MEDIUM', 'competition_level_unknown', 'age_tier_181-365', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'freshness_tier_0-30', 'freshness_tier_181+', 'freshness_tier_31-90', 'freshness_tier_91-180']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `word_count` / `char_count` | content length | filled -1 (25.7% missing, confirmed ML-04-style check below) | yes, static |
| `search_volume` / `competition` / `cpc` | keyword-level SEO metrics | filled -1 where missing | yes, static |
| `content_age_days` | how old the page is | none missing | yes, static |
| `days_since_last_update` | staleness | none missing | yes, static |
| `content_type` / `main_intent` | what kind of page, what intent | one-hot, "unknown" bucket for nulls | yes, static |
| `competition_level` | bucketed competition | one-hot, "unknown" bucket | yes, static |
| `age_tier` / `freshness_tier` | bucketed versions of age/staleness | one-hot | yes, static |
| `impressions_prev_30d` / `clicks_prev_30d` / `sessions_prev_30d` | traffic in the 30 days BEFORE the label's comparison window | filled -1 | yes, this window ends before the label's window begins |

**Deliberately excluded from this table**, not forgotten: `trend_direction`, `trend_pct` (label-derived), `impressions_last_30d`/`clicks_last_30d`/`sessions_last_30d` (the label's own window), and every 90-day aggregate (`impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`). Full reasoning for each is in Section 4.

In [2]:
missing_rates = df[numeric_cols].isna().mean().sort_values(ascending=False)
print("missing rate for numeric features actually used:")
print(missing_rates[missing_rates > 0])

missing rate for numeric features actually used:
word_count       0.256633
char_count       0.256633
search_volume    0.082267
competition      0.082267
cpc              0.082267
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Test 1, the confession test**: train once on the honest feature set, once with a deliberately re-added known-leaky column (`trend_pct`), and watch the score. Per the skill: a collapse from ~1.0 to a real, modest number when the suspect is removed is exactly what should happen if the honest set is actually clean.

**Test 2, window overlap, quantified not just asserted**: compare correlation with the label across `_last_30d` vs `_prev_30d` vs `_90d` versions of the same metric. If `_last_30d` consistently correlates harder, that's the window-overlap risk showing up as a number, not just a rule I'm trusting blindly.

**Test 3, grouped vs random split**: split by `client_id` (GroupKFold) instead of randomly, and report the gap. A model that only looks good on a random split was partly memorizing clients it already saw.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# --- Test 1: confession test ---
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

Xtr, Xte, ytr, yte = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=2000)
honest_model.fit(Xtr, ytr)
auc_honest = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])
print(f"honest AUC (leakage-safe features only): {auc_honest:.3f}")

X_leaky = X_scaled.copy()
X_leaky["trend_pct_LEAKY"] = (df["trend_pct"].fillna(0) - df["trend_pct"].fillna(0).mean()) / df["trend_pct"].fillna(0).std()
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=2000)
leaky_model.fit(Xtr2, ytr2)
auc_leaky = roc_auc_score(yte2, leaky_model.predict_proba(Xte2)[:, 1])
print(f"AUC WITH deliberate leak (trend_pct added back in): {auc_leaky:.3f}")
print(f"-> the jump from {auc_honest:.3f} to {auc_leaky:.3f} is the confession. Removing it recovers the honest number.")

# --- Test 2: window overlap, quantified ---
print("\ncorrelation with is_declining_label, by window:")
for base in ["impressions", "clicks", "sessions"]:
    for window in ["last_30d", "prev_30d", "90d"]:
        col = f"{base}_{window}"
        print(f"  {col}: {df[col].corr(df['is_declining_label']):.3f}")

# --- Test 3: grouped vs random split ---
gkf = GroupKFold(n_splits=5)
grouped_scores = cross_val_score(LogisticRegression(max_iter=2000), X_scaled, y,
                                   groups=df["client_id"], cv=gkf, scoring="roc_auc")
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_scores = cross_val_score(LogisticRegression(max_iter=2000), X_scaled, y, cv=skf, scoring="roc_auc")

print(f"\nrandom split AUC (mean of 5 folds): {random_scores.mean():.3f}")
print(f"grouped-by-client split AUC (mean of 5 folds): {grouped_scores.mean():.3f}")
print(f"gap: {random_scores.mean() - grouped_scores.mean():.3f}")

honest AUC (leakage-safe features only): 0.642
AUC WITH deliberate leak (trend_pct added back in): 1.000
-> the jump from 0.642 to 1.000 is the confession. Removing it recovers the honest number.

correlation with is_declining_label, by window:
  impressions_last_30d: -0.094
  impressions_prev_30d: 0.004
  impressions_90d: -0.018
  clicks_last_30d: -0.072
  clicks_prev_30d: -0.029
  clicks_90d: -0.040
  sessions_last_30d: -0.064
  sessions_prev_30d: -0.023
  sessions_90d: -0.023

random split AUC (mean of 5 folds): 0.644
grouped-by-client split AUC (mean of 5 folds): 0.551
gap: 0.093


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction`, `trend_pct` | Label-derived, confirmed in ML-03: `is_declining_label` is computed directly from `trend_direction`. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | This is one side of the label's own last-30-vs-prev-30 comparison, not a predictor of it. |
| `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions` | 90-day aggregates overlap the label's trailing window; Section 3's correlation check shows `_last_30d` consistently correlates harder with the label than `_prev_30d`, the window-overlap risk is real, not hypothetical, even if modest in size here. |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier` | All derived from the same 90-day window as above; no `_prev_30d`-only version of these exists in this dataset, so there's no safe sub-window to fall back to. Excluded rather than used with a caveat. |
| `content_id`, `client_id` | Identifiers, used only for grouping and the client-level split test, never as model inputs. |
| `provider_used`, `model_used` | Describe how the content was originally produced (which AI tool/model), not a property of its current performance, kept out because they're closer to a production/process detail than a genuine performance signal, and 71% of `provider_used` is missing besides. |

**Net effect**: several columns that looked like the most obviously useful features going in (`avg_position`, `ctr`) turned out to be unusable without a safe earlier-window version. The honest feature set is smaller and weaker (AUC 0.625) than the leaky one (AUC 0.998) on purpose, that gap is the entire point of this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.